In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Reshape
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

IMG_SIZE = (224,224)
BATCH_SIZE = 4




train_dir = "few_shot_dataset/train"
val_dir   = "few_shot_dataset/val"
test_dir  = "few_shot_dataset/test"


In [2]:

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    fill_mode='nearest',
    horizontal_flip=True,
    shear_range=0.1,
    brightness_range=[0.8,1.2],
    channel_shift_range=0.2
)


In [3]:
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)


train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 55 images belonging to 6 classes.
Found 33 images belonging to 6 classes.
Found 50 images belonging to 6 classes.


In [4]:

conv_base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [5]:
conv_base.trainable = True

set_trainable = False

for layer in conv_base.layers:
    if layer.name == 'block_14_expand':
        set_trainable = True

    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False
# ==================================
# MODEL INPUT
# ==================================
inputs = tf.keras.Input(shape=(224,224,3))

x = conv_base(inputs)

In [6]:
# ==================================
# CNN FEATURE MAP -> SEQUENCE
# shape = (7,7,1280) => (49,1280)
# ==================================
x = Reshape((49,1280))(x)

# ==================================
# TRANSFORMER BLOCK
# ==================================
attn = MultiHeadAttention(
    num_heads=4,
    key_dim=128
)(x, x)

attn = Dense(1280)(attn)

x = Add()([x, attn])
x = LayerNormalization()(x)

ff = Dense(1280, activation='relu')(x)
x = Add()([x, ff])
x = LayerNormalization()(x)

# ==================================
# CLASSIFIER HEAD
# ==================================
x = GlobalAveragePooling1D()(x)
x = Dropout(0.6)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(6, activation='softmax')(x)

model = Model(inputs, outputs)

In [7]:
# ==================================
# COMPILE
# ==================================
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),
    
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=3,
        factor=0.3
    )
]


In [15]:
# TRAIN
# ==================================
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=300,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 250ms/step - accuracy: 0.7455 - loss: 0.6025 - val_accuracy: 0.7273 - val_loss: 0.5269 - learning_rate: 2.7000e-06
Epoch 2/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 249ms/step - accuracy: 0.8364 - loss: 0.4255 - val_accuracy: 0.7273 - val_loss: 0.5296 - learning_rate: 2.7000e-06
Epoch 3/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 267ms/step - accuracy: 0.8727 - loss: 0.4785 - val_accuracy: 0.7576 - val_loss: 0.5316 - learning_rate: 2.7000e-06
Epoch 4/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 259ms/step - accuracy: 0.8727 - loss: 0.3530 - val_accuracy: 0.7576 - val_loss: 0.5287 - learning_rate: 8.1000e-07
Epoch 5/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 268ms/step - accuracy: 0.9273 - loss: 0.3132 - val_accuracy: 0.7879 - val_loss: 0.5240 - learning_rate: 8.1000e-07


In [16]:
# TEST
# ==================================
loss, acc = model.evaluate(test_data)

print("Test Accuracy:", acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.7800 - loss: 0.6801
Test Accuracy: 0.7799999713897705
